# 单个词报告

In [1]:
import pandas as pd
import json

from collections import defaultdict

In [2]:
import sys
sys.path.append('..')

## 原始数据

In [3]:
df_raw = pd.read_excel('data/mayday_songs.xlsx', sheet_name='Sheet1')
df_raw['song_name'] = df_raw['song_name'].astype(str)
df_raw

,album_order,album_id,album_name,album_type,release_date,song_order,song_id,song_name,album_fixed,has_lyric,is_duplicate
0,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑,1,0
1,1,38315,第一张创作专辑,录音室专辑,1999-07-07,2,386927,拥抱,第一张创作专辑,1,0
2,1,38315,第一张创作专辑,录音室专辑,1999-07-07,3,386929,透露,第一张创作专辑,1,0
3,1,38315,第一张创作专辑,录音室专辑,1999-07-07,4,386930,生活,第一张创作专辑,1,0
4,1,38315,第一张创作专辑,录音室专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑,1,0
...,...,...,...,...,...,...,...,...,...,...,...
185,11,2740205,步步 自选作品辑,精选辑,2013-12-30,26,28181124,雌雄同体,步步 自选作品辑,1,1
186,11,2740205,步步 自选作品辑,精选辑,2013-12-30,27,28181126,生命有一种绝对,步步 自选作品辑,1,1
187,11,2740205,步步 自选作品辑,精选辑,2013-12-30,28,28181128,诺亚方舟,步步 自选作品辑,1,1
188,11,2740205,步步 自选作品辑,精选辑,2013-12-30,29,28181130,我心中尚未崩坏的地方,步步 自选作品辑,1,1


In [4]:
df_lyric_raw = pd.read_csv('output/mayday_lyric_word.csv')
df_lyric_raw

,song_id,word,pos,freq,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate
0,386925,我,r,18,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0
1,386925,好想,v,18,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0
2,386925,那么,r,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0
3,386925,多,m,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0
4,386925,的,uj,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13130,28181110,长大,v,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0
13131,28181110,难道,d,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0
13132,28181110,人,n,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0
13133,28181110,必经,d,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0


In [5]:
# 曲目，专辑添加唯一id
df_lyric = df_lyric_raw.copy()
df_lyric['song_id_unique'] = 'song' + df_lyric['song_id'].astype(str)
df_album_fixed = df_lyric[['album_id', 'album_fixed']].drop_duplicates(subset=['album_fixed'], keep='first').reset_index(drop=True)
df_album_fixed['album_id_unique'] = 'album' + df_album_fixed['album_id'].astype(int).astype(str)
df_album_fixed = df_album_fixed[['album_id_unique', 'album_fixed']]
df_lyric= df_lyric.merge(df_album_fixed, on='album_fixed', how='left')
df_lyric


,song_id,word,pos,freq,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate,song_id_unique,album_id_unique
0,386925,我,r,18,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,song386925,album38315
1,386925,好想,v,18,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,song386925,album38315
2,386925,那么,r,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,song386925,album38315
3,386925,多,m,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,song386925,album38315
4,386925,的,uj,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,song386925,album38315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13130,28181110,长大,v,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,song28181110,album2740205
13131,28181110,难道,d,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,song28181110,album2740205
13132,28181110,人,n,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,song28181110,album2740205
13133,28181110,必经,d,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,song28181110,album2740205


# 词分析

In [26]:
# 词
word = "时间"
df_word = df_lyric[df_lyric['word'] == word]
df_word

,song_id,word,pos,freq,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate,song_id_unique,album_id_unique
245,386930,时间,n,3,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,4.0,生活,第一张创作专辑,1.0,0.0,song386930,album38315
2018,386691,时间,n,2,3.0,38297.0,人生海海,录音室专辑,2001-07-06,1.0,一颗苹果,人生海海,1.0,0.0,song386691,album38297
4126,386476,时间,n,1,4.0,38276.0,时光机,录音室专辑,2003-11-11,12.0,在这一秒,时光机,1.0,0.0,song386476,album38276
4275,386482,时间,n,2,4.0,38276.0,时光机,录音室专辑,2003-11-11,14.0,王子面,时光机,1.0,0.0,song386482,album38276
4546,386173,时间,n,1,5.0,38259.0,神的孩子都在跳舞,录音室专辑,2004-11-05,1.0,孙悟空,神的孩子都在跳舞,1.0,0.0,song386173,album38259
5072,386185,时间,n,1,5.0,38259.0,神的孩子都在跳舞,录音室专辑,2004-11-05,7.0,回来吧,神的孩子都在跳舞,1.0,0.0,song386185,album38259
7577,22198023,时间,n,1,7.0,38235.0,后青春期的诗,录音室专辑,2008-10-23,10.0,如烟,后青春期的诗,1.0,0.0,song22198023,album38235
8217,22197001,时间,n,1,8.0,2040001.0,第二人生 (明日版),录音室专辑,2011-12-16,3.0,星空,第二人生,1.0,0.0,song22197001,album2040001
8306,22197003,时间,n,3,8.0,2040001.0,第二人生 (明日版),录音室专辑,2011-12-16,4.0,洗衣机,第二人生,1.0,0.0,song22197003,album2040001
8504,22197004,时间,n,2,8.0,2040001.0,第二人生 (明日版),录音室专辑,2011-12-16,5.0,三个傻瓜,第二人生,1.0,0.0,song22197004,album2040001


In [27]:
song_ids = df_word['song_id'].unique()
song_ids

array([    386930,     386691,     386476,     386482,     386173,
           386185,   22198023,   22197001,   22197003,   22197004,
         22197007,   22197008,   22197009, 2008204010,  417953659,
        422094253,  422094257])

In [28]:
with open('output/mayday_lyric_260124.json', 'r', encoding='utf-8') as f:
    lyric_raw = json.load(f)
lyric_dict = defaultdict(dict)
for i in lyric_raw:
    if i:
        lyric_dict[i['song_id']] = i
        lyric_dict[i['song_id']]['lyric_splited'] = i['歌词'].split('。')
lyric_dict

defaultdict(dict,
            {'386925': {'歌名': '疯狂世界',
              'song_id': '386925',
              '作词': '五月天 阿信',
              '作曲': '五月天 阿信',
              '编曲': '五月天',
              '歌词': '如果说了后悔，是不是一切就能倒退。回忆多么美，活着多么狼狈。为什么这个世界，总要叫人尝伤悲。我不能了解，也不想了解。我好想好想飞。逃离这个疯狂世界。那么多苦，那么多累。那么多莫名的泪水。我好想好想飞。逃离这个疯狂的世界。如果是你，发现了我。也别将我挽回。想了你一整夜，再也想不起你的脸。你是一种感觉，写在夏夜晚风里面。青春是挽不回的水，转眼消失在指间。用力的浪费，再用力的后悔。我好想好想飞。我好想好想飞。逃离这个疯狂世界。那么多苦，那么多累。那么多莫名的泪水。我好想好想飞，逃离这个疯狂的世界。如果是你，发现了我。也别将我挽回。~~~~~~。我好想好想飞。逃离这个疯狂世界。那么多苦，那么多累。那么多莫名的泪水。我好想好想飞。逃离这个疯狂的世界。如果是你，发现了我。也别将我挽回。我好想好想飞。逃离这个疯狂世界。那么多苦，那么多累。那么多莫名的伤悲。我好想好想飞。逃离这个疯狂的世界。如果是你，发现了我。也别将我挽回。',
              'lyric_splited': ['如果说了后悔，是不是一切就能倒退',
               '回忆多么美，活着多么狼狈',
               '为什么这个世界，总要叫人尝伤悲',
               '我不能了解，也不想了解',
               '我好想好想飞',
               '逃离这个疯狂世界',
               '那么多苦，那么多累',
               '那么多莫名的泪水',
               '我好想好想飞',
               '逃离这个疯狂的世界',
               '如果是你，发现了我',
               '也别将我挽回',
               '想了你一整夜，再也想不起

In [ ]:
word_dict = defaultdict(list)
for song_id in song_ids:
    word_lyric = []
    song_id = str(song_id)
    for i in lyric_dict[song_id]['lyric_splited']:
        if word in i:
            word_lyric.append(i)
    word_dict[song_id] = list(set(word_lyric))
word_dict

defaultdict(list,
            {'386930': ['那些笑和眼泪，没有时间说再见', '珍惜的浪费时间，换来了的生命的缺'],
             '386691': ['那时间忘记挽留最美时候', '时间如果可以倒流我想'],
             '386476': ['时间它虽然重要，也不必抓的太牢'],
             '386482': ['用意大利式的，浪费时间，美好事物瞬间', '时间倒带闭上眼睛倒带时间'],
             '386173': ['好色本性多隽永，好像时间从来没走'],
             '386185': ['所有的人都说时间是魔法'],
             '22198023': ['时间是贼偷走一切'],
             '22197001': ['时间偷走初衷，只留下了苦衷'],
             '22197003': ['多少年了，旋转又旋转，时间一眨眼过去'],
             '22197004': ['傻不傻瓜，时间一样追杀'],
             '22197007': ['时间都停了他们都回来了', '会不会有一天时间真的能倒退'],
             '22197008': ['想逆转时间，回到，最开始，有你的世界'],
             '22197009': ['再没有时间，能去延后'],
             '2008204010': ['时间都停了，他们都回来了'],
             '417953659': ['无数时间线无尽可能性'],
             '422094253': ['忘了时间有脚', '时间的电影，结局才知道'],
             '422094257': ['时间就是，最巨大的谎']})

In [10]:
len(word_dict)

51

In [11]:
# 个别修正
word_dict_mannul_fix = {
    '386925': ['为什么这个世界，总要叫人尝伤悲', '我好想好想飞，逃离这个疯狂世界', '我好想好想飞，逃离这个疯狂的世界'],
    '422094256': [
        '你问我全世界是哪里最美，答案是你身边', '平凡的我们，也将回到，平凡的世界',
        '每个梦都，像任意门，往不同世界，而你的故事，现在正是起点', '我们曾走过，无数地方，和无尽岁月，搭着肩环游，无法遗忘，的光辉世界'
    ],
    '385976': [
        '世界若是，那么大，为何我要忘你，无处逃',
        '世界若是，那么小，为何我的真心，你听不到',
    ],
    '385787': ['这世界笑了，于是你合群的一起笑了'],
    '422094253': ['世界再大不过，你我凝视的微笑'],
    '385887': ['我来到这个世界，这个人生为你而生存'],
    '386479': ['想要征服的世界，始终都没有改变'],
    '386717': ['就算是整个世界，把我抛弃', '就算真的整个世界，把我抛弃'],
    '22198023': ['有没有那么一个世界，永远不天黑'],
    '385891': ['不管世界变得怎麽样，只要有你就会是天堂'],
    '385798': ['一生能有几次，跟世界宣战'],
}

In [12]:
for k, v in word_dict_mannul_fix.items():
    word_dict[k] = v
word_dict

defaultdict(list,
            {'386925': ['为什么这个世界，总要叫人尝伤悲',
              '我好想好想飞，逃离这个疯狂世界',
              '我好想好想飞，逃离这个疯狂的世界'],
             '386927': ['纷乱世界的不了解'],
             '386930': ['活在疯狂世界，活在美好的明天'],
             '386931': ['教我勇敢地挑战全世界', '这世界全部的漂亮'],
             '386939': ['世界哪会颠三倒四'],
             '386942': ['汝予我的世界只有一条路', '哪会世界拢无声音'],
             '386846': ['纵然是世界辽阔，外面的精彩好多'],
             '386848': ['开伤济气力，在这个世界，有小可仔无彩'],
             '386851': ['扰乱我原本平静的世界'],
             '386854': ['梦中的许个世界'],
             '386863': ['多希望世界就快要消失，好让我俩享受没有明天', '有一个漩涡，让我们躲避，真实世界，有什么刺激'],
             '386866': ['没有关系，你的世界，就让你拥有'],
             '386691': ['这世界给我的幽默'],
             '386700': ['世界就睡在梦里'],
             '386702': ['世界欲共汝捙拚'],
             '386715': ['世界被你掌握', '世界的纯真此刻为你有迷惑', '我以为世界是座宁静的宇宙'],
             '386717': ['就算是整个世界，把我抛弃', '就算真的整个世界，把我抛弃'],
             '386443': ['人在江湖心不由己这世界又烦又粘腻'],
             '386445': ['你要是落泪滴，世界都要下雨'],
             '386451': ['世界很大，（而我们应该长大）'],
 

In [13]:
df_word = df_word.sort_values(by=['release_date', 'song_order'], )
df_word

,song_id,word,pos,freq,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate,song_id_unique,album_id_unique
6,386925,世界,n,9,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,song386925,album38315
129,386927,世界,n,1,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,2.0,拥抱,第一张创作专辑,1.0,0.0,song386927,album38315
232,386930,世界,n,3,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,4.0,生活,第一张创作专辑,1.0,0.0,song386930,album38315
419,386931,世界,n,1,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,5.0,爱情的模样,第一张创作专辑,1.0,0.0,song386931,album38315
718,386939,世界,n,1,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,9.0,HoSee,第一张创作专辑,1.0,0.0,song386939,album38315
802,386942,世界,n,5,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,10.0,黑白讲,第一张创作专辑,1.0,0.0,song386942,album38315
1211,386846,世界,n,2,2.0,38308.0,爱情万岁,录音室专辑,2000-07-07,3.0,明白,爱情万岁,1.0,0.0,song386846,album38308
1329,386848,世界,n,1,2.0,38308.0,爱情万岁,录音室专辑,2000-07-07,4.0,心中无别人,爱情万岁,1.0,0.0,song386848,album38308
1409,386851,世界,n,2,2.0,38308.0,爱情万岁,录音室专辑,2000-07-07,5.0,有你的将来,爱情万岁,1.0,0.0,song386851,album38308
1438,386854,世界,n,3,2.0,38308.0,爱情万岁,录音室专辑,2000-07-07,6.0,憨人,爱情万岁,1.0,0.0,song386854,album38308


In [14]:
res_df = df_word[['song_id', 'song_name', 'album_fixed', 'release_date']].copy().reset_index(drop=True)
res_df['song_id'] = res_df['song_id'].astype(str)
res_df

,song_id,song_name,album_fixed,release_date
0,386925,疯狂世界,第一张创作专辑,1999-07-07
1,386927,拥抱,第一张创作专辑,1999-07-07
2,386930,生活,第一张创作专辑,1999-07-07
3,386931,爱情的模样,第一张创作专辑,1999-07-07
4,386939,HoSee,第一张创作专辑,1999-07-07
5,386942,黑白讲,第一张创作专辑,1999-07-07
6,386846,明白,爱情万岁,2000-07-07
7,386848,心中无别人,爱情万岁,2000-07-07
8,386851,有你的将来,爱情万岁,2000-07-07
9,386854,憨人,爱情万岁,2000-07-07


In [15]:
res_df['lyric'] = res_df['song_id'].map(word_dict)
res_df

,song_id,song_name,album_fixed,release_date,lyric
0,386925,疯狂世界,第一张创作专辑,1999-07-07,"[为什么这个世界，总要叫人尝伤悲, 我好想好想飞，逃离这个疯狂世界, 我好想好想飞，逃离这个..."
1,386927,拥抱,第一张创作专辑,1999-07-07,[纷乱世界的不了解]
2,386930,生活,第一张创作专辑,1999-07-07,[活在疯狂世界，活在美好的明天]
3,386931,爱情的模样,第一张创作专辑,1999-07-07,"[教我勇敢地挑战全世界, 这世界全部的漂亮]"
4,386939,HoSee,第一张创作专辑,1999-07-07,[世界哪会颠三倒四]
5,386942,黑白讲,第一张创作专辑,1999-07-07,"[汝予我的世界只有一条路, 哪会世界拢无声音]"
6,386846,明白,爱情万岁,2000-07-07,[纵然是世界辽阔，外面的精彩好多]
7,386848,心中无别人,爱情万岁,2000-07-07,[开伤济气力，在这个世界，有小可仔无彩]
8,386851,有你的将来,爱情万岁,2000-07-07,[扰乱我原本平静的世界]
9,386854,憨人,爱情万岁,2000-07-07,[梦中的许个世界]


In [16]:
res_dict = res_df.to_dict('records')
res_dict

[{'song_id': '386925',
  'song_name': '疯狂世界',
  'album_fixed': '第一张创作专辑',
  'release_date': '1999-07-07',
  'lyric': ['为什么这个世界，总要叫人尝伤悲', '我好想好想飞，逃离这个疯狂世界', '我好想好想飞，逃离这个疯狂的世界']},
 {'song_id': '386927',
  'song_name': '拥抱',
  'album_fixed': '第一张创作专辑',
  'release_date': '1999-07-07',
  'lyric': ['纷乱世界的不了解']},
 {'song_id': '386930',
  'song_name': '生活',
  'album_fixed': '第一张创作专辑',
  'release_date': '1999-07-07',
  'lyric': ['活在疯狂世界，活在美好的明天']},
 {'song_id': '386931',
  'song_name': '爱情的模样',
  'album_fixed': '第一张创作专辑',
  'release_date': '1999-07-07',
  'lyric': ['教我勇敢地挑战全世界', '这世界全部的漂亮']},
 {'song_id': '386939',
  'song_name': 'HoSee',
  'album_fixed': '第一张创作专辑',
  'release_date': '1999-07-07',
  'lyric': ['世界哪会颠三倒四']},
 {'song_id': '386942',
  'song_name': '黑白讲',
  'album_fixed': '第一张创作专辑',
  'release_date': '1999-07-07',
  'lyric': ['汝予我的世界只有一条路', '哪会世界拢无声音']},
 {'song_id': '386846',
  'song_name': '明白',
  'album_fixed': '爱情万岁',
  'release_date': '2000-07-07',
  'lyric': ['纵然是世界辽阔，外面的精彩好多'

In [17]:
with open('output/single_word_data.json', 'w', encoding='utf-8') as f:
    json.dump(res_dict, f, ensure_ascii=False, indent=4)